# Tutorial 1: 16S rRNA Gene Tree Annotation with PyiTOL

**16S rRNA 基因树注释教程 -- 使用 PyiTOL 自动生成 iTOL 模板文件**

This notebook demonstrates how to use PyiTOL's template generation functions to create
iTOL-compatible dataset files for annotating a 16S rRNA phylogenetic tree. We will:

1. Generate a synthetic phylogenetic tree with 50 leaves using DendroPy
2. Create synthetic taxonomy metadata (Phylum, Class, Order, Family, Genus)
3. Generate color-strip, heatmap, and bar chart iTOL templates
4. Show how to upload the tree and templates to iTOL (simulated)

本教程展示如何使用 PyiTOL 的模板生成功能为 16S rRNA 系统发育树创建 iTOL 数据集文件。

---

**Prerequisites / 前置条件:**
- `pyitol` (install via `pip install -e .` from the project root)
- `dendropy` (included in pyitol dependencies)
- `pandas` (included in pyitol dependencies)

## 1. Setup and Imports

导入所需的库。

In [ ]:
import os
import random
import tempfile
from pathlib import Path

import dendropy
import pandas as pd

# PyiTOL template generators
from pyitol.templates.generator import (
    generate_color_strip_template,
    generate_heatmap_template,
    generate_simple_bar_template,
    generate_branch_template,
)
from pyitol.templates.presets.nature import get_nature_palette
from pyitol.templates.presets.colorblind import get_colorblind_palette

## 2. Generate a Synthetic Phylogenetic Tree

We create a random 50-leaf ultrametric tree using DendroPy's `purebirth` birth-death model.
The leaf names follow the pattern `Seq_001` through `Seq_050`.

使用 DendroPy 的纯出生模型生成一棵包含 50 个叶节点的合成系统发育树。

In [ ]:
# Set random seed for reproducibility
random.seed(42)

# Create a pure-birth tree with 50 tips
tree = dendropy.Tree.purebirth_taxa_tree(num_leaves=50)

# Rename leaves to Seq_001 .. Seq_050
for i, leaf in enumerate(tree.leaf_node_iter(), 1):
    leaf.taxon.label = f"Seq_{i:03d}"

# Write to a temporary Newick file
tmpdir = Path(tempfile.mkdtemp(prefix="pyitol_tutorial_"))
tree_path = tmpdir / "16s_rrna_tree.nwk"
tree.write_to_path(str(tree_path), schema="newick")

print(f"Tree written to: {tree_path}")
print(f"Number of leaves: {len(tree.leaf_nodes())}")
print(f"First 10 leaf names: {[leaf.taxon.label for leaf in tree.leaf_node_iter()][:10]}")

## 3. Create Synthetic Taxonomy Metadata

We simulate a realistic 16S rRNA taxonomy table with the following columns:
- `id`: Sequence identifier (matches tree leaf names)
- `Phylum`: Bacterial phylum (e.g., Proteobacteria, Firmicutes)
- `Class`: Taxonomic class
- `Order`: Taxonomic order
- `Family`: Taxonomic family
- `Genus`: Taxonomic genus
- `Abundance`: Simulated relative abundance (numeric, for heatmap/bar)
- `GC_Content`: Simulated GC content percentage (numeric)

创建模拟的 16S rRNA 分类学元数据表。

In [ ]:
# Define a realistic taxonomy hierarchy
taxonomy_hierarchy = {
    "Proteobacteria": {
        "Gammaproteobacteria": {
            "Enterobacterales": {
                "Enterobacteriaceae": ["Escherichia", "Salmonella", "Klebsiella"],
                "Yersiniaceae": ["Serratia"],
            },
            "Pseudomonadales": {
                "Pseudomonadaceae": ["Pseudomonas"],
                "Moraxellaceae": ["Acinetobacter"],
            },
        },
        "Betaproteobacteria": {
            "Burkholderiales": {
                "Burkholderiaceae": ["Burkholderia", "Ralstonia"],
            },
        },
    },
    "Firmicutes": {
        "Bacilli": {
            "Bacillales": {
                "Bacillaceae": ["Bacillus"],
                "Staphylococcaceae": ["Staphylococcus"],
            },
            "Lactobacillales": {
                "Lactobacillaceae": ["Lactobacillus"],
                "Streptococcaceae": ["Streptococcus"],
            },
        },
    },
    "Actinobacteria": {
        "Actinomycetia": {
            "Corynebacteriales": {
                "Corynebacteriaceae": ["Corynebacterium"],
                "Mycobacteriaceae": ["Mycobacterium"],
            },
        },
    },
}

# Flatten the hierarchy into a list of (Phylum, Class, Order, Family, Genus) tuples
taxonomy_records = []
for phylum, classes in taxonomy_hierarchy.items():
    for cls, orders in classes.items():
        for order, families in orders.items():
            for family, genera in families.items():
                for genus in genera:
                    taxonomy_records.append((phylum, cls, order, family, genus))

# Assign taxonomy to each leaf, distributing roughly evenly
leaf_names = [leaf.taxon.label for leaf in tree.leaf_node_iter()]
records = []
for i, name in enumerate(leaf_names):
    rec = taxonomy_records[i % len(taxonomy_records)]
    records.append({
        "id": name,
        "Phylum": rec[0],
        "Class": rec[1],
        "Order": rec[2],
        "Family": rec[3],
        "Genus": rec[4],
        "Abundance": round(random.uniform(0.1, 15.0), 2),
        "GC_Content": round(random.uniform(35.0, 65.0), 1),
    })

taxonomy_df = pd.DataFrame(records)

# Save to CSV
taxonomy_path = tmpdir / "taxonomy.csv"
taxonomy_df.to_csv(taxonomy_path, index=False)

print(f"Taxonomy saved to: {taxonomy_path}")
print(f"Shape: {taxonomy_df.shape}")
taxonomy_df.head(10)

## 4. Generate Color-Strip Template

The color-strip template maps each Phylum to a color and displays it as a colored strip
next to the tree. This is one of the most commonly used iTOL annotation types.

颜色条模板将每个 Phylum（门）映射到一种颜色，并在树旁显示为彩色条带。
这是最常用的 iTOL 注释类型之一。

In [ ]:
# Get a colorblind-friendly palette for phyla
phyla = sorted(taxonomy_df["Phylum"].unique())
palette = get_colorblind_palette("tol_bright", n=len(phyla))
phylum_colors = dict(zip(phyla, palette))

print("Phylum color mapping:")
for phylum, color in phylum_colors.items():
    print(f"  {phylum}: {color}")

In [ ]:
# Generate color-strip template for Phylum
color_strip_path = tmpdir / "phylum_color_strip.txt"

generate_color_strip_template(
    output_path=color_strip_path,
    taxonomy_path=taxonomy_path,
    tree_path=tree_path,
    column="Phylum",
    colors=phylum_colors,
    label="Phylum",
    strip_width="60",
)

print(f"Color-strip template written to: {color_strip_path}")
print("--- Preview (first 30 lines) ---")
print(color_strip_path.read_text()[:800])

In [ ]:
# Also generate a branch coloring template by Phylum
branch_path = tmpdir / "phylum_branch_colors.txt"

generate_branch_template(
    output_path=branch_path,
    taxonomy_path=taxonomy_path,
    tree_path=tree_path,
    column="Phylum",
    colors=phylum_colors,
    label="Phylum_branches",
)

print(f"Branch-coloring template written to: {branch_path}")
print("--- Preview (first 20 lines) ---")
print(branch_path.read_text()[:600])

## 5. Generate Heatmap Template

A heatmap template visualizes numeric data (Abundance, GC_Content) as a color gradient
alongside the tree. We use a red-white-blue gradient.

热力图模板将数值数据（丰度、GC 含量）以颜色渐变的形式显示在树旁。

In [ ]:
heatmap_path = tmpdir / "abundance_heatmap.txt"

generate_heatmap_template(
    output_path=heatmap_path,
    taxonomy_path=taxonomy_path,
    tree_path=tree_path,
    value_columns=["Abundance", "GC_Content"],
    color_gradient=["#d73027", "#ffffbf", "#4575b4"],
    label="Abundance_GC",  
    column_labels={"Abundance": "Rel. Abundance", "GC_Content": "GC %"},
)

print(f"Heatmap template written to: {heatmap_path}")
print("--- Preview (first 30 lines) ---")
print(heatmap_path.read_text()[:800])

## 6. Generate Bar Chart Template

A simple bar chart template displays a single numeric value (Abundance) as horizontal bars
next to each leaf.

简单柱状图模板将单一数值（丰度）显示为每个叶节点旁的水平条。

In [ ]:
bar_path = tmpdir / "abundance_bar.txt"

generate_simple_bar_template(
    output_path=bar_path,
    taxonomy_path=taxonomy_path,
    tree_path=tree_path,
    value_column="Abundance",
    label="Abundance_bar",
    bar_color="#3c5484",
    bar_width="80",
)

print(f"Bar chart template written to: {bar_path}")
print("--- Preview (first 25 lines) ---")
print(bar_path.read_text()[:600])

## 7. Review Generated Files

Let's list all the template files we've generated.

查看所有生成的模板文件。

In [ ]:
generated_files = list(tmpdir.glob("*.txt")) + list(tmpdir.glob("*.nwk")) + list(tmpdir.glob("*.csv"))
print("Generated files:")
for f in sorted(generated_files):
    size = f.stat().st_size
    print(f"  {f.name:30s}  {size:>6d} bytes")

print(f"\nAll files are in: {tmpdir}")

## 8. Upload to iTOL (Simulated)

In a real workflow, you would use `ITOLAPIClient` to upload the tree and templates to iTOL.
Below we show the exact code, but the actual API call is commented out since it requires
a valid API key.

在实际工作流中，您可以使用 `ITOLAPIClient` 将树和模板上传到 iTOL。
下面展示了完整的代码，但实际的 API 调用已注释掉（需要有效的 API Key）。

In [ ]:
from pyitol.api.client import ITOLAPIClient

# --- Simulated upload workflow ---
# In a real scenario, you would do:
#
# client = ITOLAPIClient(api_key_file=".itolapi.key")
# tree_id = client.upload(
#     tree_file=str(tree_path),
#     template_files=[
#         str(color_strip_path),
#         str(branch_path),
#         str(heatmap_path),
#         str(bar_path),
#     ],
#     force=True,
# )
# print(f"Tree uploaded. iTOL tree ID: {tree_id}")
# print(f"View at: https://itol.embl.de/tree/{tree_id}")
#
# # Export as PDF
# result = client.export([tree_id], format="pdf", output_dir=str(tmpdir))
# print(f"Exported: {result}")

# For this tutorial, we simulate the response:
print("[Simulated] Upload would send:")
print(f"  Tree file:    {tree_path.name}")
print(f"  Templates:    {[f.name for f in [color_strip_path, branch_path, heatmap_path, bar_path]]}")
print(f"  Tree ID:      simulated_tree_id_12345")
print(f"  View URL:     https://itol.embl.de/tree/simulated_tree_id_12345")

## 9. Summary

In this tutorial we demonstrated the core PyiTOL workflow for 16S rRNA tree annotation:

1. **Tree generation**: Used DendroPy to create a synthetic 50-leaf tree
2. **Taxonomy metadata**: Created a CSV with hierarchical taxonomy and numeric data
3. **Color-strip template**: `generate_color_strip_template()` for Phylum-level coloring
4. **Branch coloring**: `generate_branch_template()` for clade-based branch colors
5. **Heatmap template**: `generate_heatmap_template()` for multi-column numeric data
6. **Bar chart template**: `generate_simple_bar_template()` for single-value bars
7. **iTOL upload**: Shown the upload workflow (simulated)

**本教程总结：**

我们展示了 PyiTOL 的 16S rRNA 基因树注释核心工作流，包括树生成、分类学元数据创建、
多种 iTOL 模板生成（颜色条、分支着色、热力图、柱状图）以及 iTOL 上传流程。

---

### Next Steps / 后续步骤

- See `02_virus_monophyly.ipynb` for monophyly analysis
- See `03_batch_workflow.ipynb` for batch processing multiple trees

In [ ]:
# Cleanup temporary directory (optional)
# Uncomment the next line to remove generated files:
# import shutil; shutil.rmtree(tmpdir)
print(f"Tutorial complete. Temporary files are in: {tmpdir}")